In [2]:
# Run only once
!pip install transformers
!pip install torch
!pip install sentencepiece
!pip install accelerate
!pip install tqdm
!pip install pandas


In [37]:
import json
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [38]:
with open("C:\\NLP\\nlp sem project\\benchmark.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))
print(data[0])


Total samples: 37473
{'id': 'ambiguous_2039', 'question': 'How old is was the oldest person to ever live?', 'answers': ['122 years, 164 days'], 'category': 'ambiguous', 'source': 'ambigqa', 'metadata': {'difficulty': None, 'bias_group': None}}


In [63]:
import json
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from tqdm import tqdm
import os

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ===== LOAD DATA =====
with open("C:\\NLP\\nlp sem project\\benchmark.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))


# ===== LOAD MODEL =====
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)


# ===== PREDICTION FUNCTION (BATCH) =====
def get_prediction_batch(questions):

    prompts = [f"Answer the question briefly:\n{q}" for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


# ===== RUN INFERENCE =====
results = []
BATCH = 8

for i in tqdm(range(0, len(data), BATCH)):

    batch = data[i:i+BATCH]

    preds = get_prediction_batch([x["question"] for x in batch])

    for item, pred in zip(batch, preds):

        results.append({
            "id": item["id"],
            "question": item["question"],      # ✅ included
            "prediction": pred,
            "category": item["category"],
            "answers": item["answers"]
        })

    # ----- checkpoint save -----
    if i % 500 == 0:
        os.makedirs("../predictions", exist_ok=True)

        with open("../predictions/flan_t5_full.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)


# ===== FINAL SAVE =====
with open("../predictions/flan_t5_full.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("FLAN-T5 FULL DONE:", len(results))


Using device: cuda
Total samples: 37473


100%|██████████| 4685/4685 [59:25<00:00,  1.31it/s]    


FLAN-T5 FULL DONE: 37473


In [65]:
import json
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm
import os

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ===== LOAD DATA =====
with open("C:\\NLP\\nlp sem project\\benchmark.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))


# ===== LOAD MODEL =====
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

# GPT2 has no pad token → use EOS as pad
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left padding for causal LM


# ===== PREDICTION FUNCTION (BATCH) =====
def get_prediction_batch(questions):

    prompts = [f"Question: {q}\nAnswer:" for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        pad_token_id=tokenizer.eos_token_id
    )

    texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    return [t.split("Answer:")[-1].strip() for t in texts]


# ===== RUN INFERENCE =====
results = []
BATCH = 8

for i in tqdm(range(0, len(data), BATCH)):

    batch = data[i:i+BATCH]

    preds = get_prediction_batch([x["question"] for x in batch])

    for item, pred in zip(batch, preds):

        results.append({
            "id": item["id"],
            "question": item["question"],     # ✅ included
            "prediction": pred,
            "category": item["category"],
            "answers": item["answers"]
        })

    # ----- checkpoint save -----
    if i % 500 == 0:
        os.makedirs("../predictions", exist_ok=True)

        with open("../predictions/gpt2_full.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)


# ===== FINAL SAVE =====
with open("../predictions/gpt2_full.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("GPT2 FULL DONE:", len(results))


Using device: cuda
Total samples: 37473


100%|██████████| 4685/4685 [26:26<00:00,  2.95it/s]


GPT2 FULL DONE: 37473


In [3]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")

print(model)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [4]:
config = model.config

print("GPT-2 CONFIGURATION")
print("Layers:", config.n_layer)
print("Hidden size:", config.n_embd)
print("Attention heads:", config.n_head)
print("Max positions:", config.n_positions)
print("Vocabulary:", config.vocab_size)


GPT-2 CONFIGURATION
Layers: 12
Hidden size: 768
Attention heads: 12
Max positions: 1024
Vocabulary: 50257


In [5]:
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

print(model)

config = model.config
print("FLAN-T5 CONFIGURATION")
print("Layers:", config.num_layers)
print("Hidden size:", config.d_model)
print("Attention heads:", config.num_heads)
print("Max positions:", config.n_positions)
print("Vocabulary:", config.vocab_size)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [1]:
import json
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from tqdm import tqdm
import os

# ===== DEVICE =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===== LOAD DATA =====
with open("C:\\NLP\\nlp sem project\\benchmark.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total samples:", len(data))

# ===== LOAD MODEL =====
model_name = "distilgpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)

# DistilGPT2 has no pad token → use EOS token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# ===== PREDICTION FUNCTION (BATCH) =====
def get_prediction_batch(questions):

    prompts = [f"Question: {q}\nAnswer:" for q in questions]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.eos_token_id
    )

    texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Extract only generated answer part
    return [t.split("Answer:")[-1].strip() for t in texts]


# ===== RUN INFERENCE =====
results = []
BATCH = 8

for i in tqdm(range(0, len(data), BATCH)):

    batch = data[i:i+BATCH]

    preds = get_prediction_batch([x["question"] for x in batch])

    for item, pred in zip(batch, preds):

        results.append({
            "id": item["id"],
            "question": item["question"],
            "prediction": pred,
            "category": item["category"],
            "answers": item["answers"]
        })

    # ----- checkpoint save -----
    if i % 500 == 0:
        os.makedirs("../predictions", exist_ok=True)

        with open("../predictions/distilgpt2_full.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

# ===== FINAL SAVE =====
with open("../predictions/distilgpt2_full.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("DistilGPT2 FULL DONE:", len(results))

Using device: cuda
Total samples: 37473


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

c:\Users\prath\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prath\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

100%|██████████| 4685/4685 [1:26:31<00:00,  1.11s/it]     


DistilGPT2 FULL DONE: 37473
